# 02 — k-Sensitivity Re-fit of the Eigenspectrum Validity Gate

**This is the execution of a pre-registered analysis.** The k set, the statistics, the two
co-diagnostics and the interpretation rule were fixed and committed to git in
`.planning/phases/02-eigenspectrum-audit-validity-gate/02-REFIT-PREREGISTRATION.md`
(commit `057b084`) **before this notebook existed and before any re-fit was run**, so that
no result produced here can be selected for being convenient.

**What this analysis may not do** (pre-registration §6, binding):

- may not revise any of the four thresholds, or add, drop or reweight a gate statistic;
- may not test a `k` outside `K_REFIT = [5, 10, 30]`;
- may not adopt a new `k*` (Rule C: a surviving candidate is *reported*, never adopted);
- may not treat a FAIL as an error to work around;
- may not invent a pass/fail threshold for the two co-diagnostics — they are descriptive and
  enter only through Rule B's qualitative test;
- must report the full table for all of `{5, 10, 15, 30}` regardless of outcome.

**Hypotheses under test.** H1 (intrinsic curvature): the negative eigenvalue tail is real
geometry and no choice of `k` removes it. H2 (kNN-graph hop inflation): the tail is a graph
artifact that shrinks as the graph densifies. They make opposite predictions about `m(k)`.

## §1. Environment and provenance

Package versions participate in the `config_key` that names every cached artifact, so a
version drift would produce a different `fit_key` and be caught by the `fit_key`
reconstruction assertion in §3 rather than silently re-fitting.

In [1]:
import gc
import json as _json
import os
import resource
import subprocess
import sys
import time
from pathlib import Path

# Make the notebook-local pu_manifold package importable exactly as 01 does (plain relative
# import, never installed, never imported from src/effdim/).
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import joblib
import numpy as np
import scipy
import scipy.linalg
import sklearn
from scipy.sparse.csgraph import connected_components
from sklearn.manifold import Isomap

from pu_manifold import cache_path, config_key, joblib_cache, json_cache, load_subsample, npz_cache

SEED = 20260729

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("=== Reproducibility header (02) ===")
print(f"python         = {sys.version.split()[0]}")
print(f"SEED           = {SEED}")
print(f"numpy          = {np.__version__}")
print(f"scipy          = {scipy.__version__}")
print(f"scikit-learn   = {sklearn.__version__}")
print(f"git commit SHA = {git_sha}")
print(f"cwd            = {NOTEBOOK_DIR}")

=== Reproducibility header (02) ===
python         = 3.14.6
SEED           = 20260729
numpy          = 2.5.1
scipy          = 1.18.0
scikit-learn   = 1.9.0
git commit SHA = f30a882
cwd            = /home/akagi/Documents/Projects/EffDim/notebooks


## §2. Pre-registered constants

`K_REFIT` is fixed at `[5, 10, 30]` by pre-registration §4.1 and is asserted, not chosen
here. k=8 and k=20 are deliberately excluded (Phase 1 dropped them under
`STAGE2_MAX_FITS=4`; adding them now, after a FAIL, would be widening the search in response
to a result). k=15 is the incumbent and is not re-fit.

In [ ]:
# --- Thresholds, fixed by pre-registration (02-REFIT-PREREGISTRATION.md §4.3, commit
# 057b084). NOT revisable here. ---
R_MAX_PASS = 0.10
M_MAX_PASS = 0.05
R_MAX_MARGINAL = 0.25
M_MAX_MARGINAL = 0.15

D_SWEEP_MAX = 40
R2_PAIR_COUNT = 200_000
R2_PAIR_SEED = SEED + 2
R2_PAIR_SEED_CHECK = SEED + 3
SYMMETRY_RTOL = 1e-10

# --- Pre-registration §4.1: the k set, fixed before any fit ran. ---
K_REFIT = [5, 10, 30]
K_INCUMBENT = 15
K_ALL = [5, 10, 15, 30]

# --- Pre-registration §4.2: everything except n_neighbors is pinned. ---
N_COMPONENTS = 18
M_STAT_INCUMBENT_PUBLISHED = 0.412071  # 02-01-SUMMARY.md, for regression-checking only
R_STAT_INCUMBENT_PUBLISHED = 0.052419
FIT_KEY_INCUMBENT = "43cf438bc944c509"

REFIT_PREREG = True  # cell-ordering anchor: must precede REFIT_COMPUTE (§4)

assert K_REFIT == [5, 10, 30], (
    f"K_REFIT={K_REFIT} is not the pre-registered set [5, 10, 30]. Pre-registration §4.1: "
    f"no k outside this set may be added without a new, separately committed "
    f"pre-registration recorded as an amendment."
)
assert K_INCUMBENT not in K_REFIT
assert sorted(K_ALL) == sorted(K_REFIT + [K_INCUMBENT])

print(f"K_REFIT      = {K_REFIT}   (pre-registration §4.1, fixed before any fit)")
print(f"K_INCUMBENT  = {K_INCUMBENT}  (not re-fit; its measured values are the baseline)")
print(f"N_COMPONENTS = {N_COMPONENTS} (held constant for comparability, §4.2)")

## §3. Subsample and the pinned fit configuration

The subsample is **loaded from the existing cache** — the same seed, the same 10,000
`row_indices`, no re-download and no re-subsample. That is proved mechanically: the cached
`subsample_20260729_*.npz` is required to exist *before* `load_subsample` is called, and its
size and mtime are asserted unchanged afterwards.

`ANALYSIS_CFG_BASE` pins every fit-parameter field (dataset, seed, n_rows, normalize,
n_components, eigen_solver). The proof that the reconstruction is exact is that
`config_key(ANALYSIS_CFG_BASE | {n_neighbors: 15})` must reproduce the incumbent
`fit_key = 43cf438bc944c509` — if any pinned field or library version differed, the hash
would differ and this cell would halt.

In [3]:
ANALYSIS_CFG_BASE = {
    "dataset": "legacysurvey_dinov3_vitb16",
    "seed": SEED,
    "n_rows": 10_000,
    "normalize": True,
    "n_neighbors": None,  # the ONLY field that varies across this analysis
    "n_components": N_COMPONENTS,
    "eigen_solver": "dense",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}

# --- Prove the subsample is a cache hit, not a fresh download/subsample -------------------
_subsample_paths = sorted(cache_path("x", "npz").parent.glob(f"subsample_{SEED}_*.npz"))
_subsample_paths = [p for p in _subsample_paths if p.stat().st_size > 50 * 1024**2]
assert len(_subsample_paths) == 1, (
    f"expected exactly one n=10,000 subsample npz in the cache, found {_subsample_paths}"
)
_sub_path = _subsample_paths[0]
_sub_stat_before = (_sub_path.stat().st_size, _sub_path.stat().st_mtime_ns)
print(f"cached subsample present before load: {_sub_path.name} "
      f"({_sub_stat_before[0] / 1024**2:.1f} MiB)")

analysis_data = load_subsample(ANALYSIS_CFG_BASE)
LS = analysis_data["legacysurvey"]
ROW_INDICES = analysis_data["row_indices"]

_sub_stat_after = (_sub_path.stat().st_size, _sub_path.stat().st_mtime_ns)
assert _sub_stat_after == _sub_stat_before, (
    "the subsample npz changed on disk during load_subsample -- this run re-subsampled "
    "rather than hitting the cache, which would break comparability with the k=15 fit."
)
print("cached subsample byte-size and mtime unchanged after load: cache HIT confirmed "
      "(no re-download, no re-subsample).")

assert LS.shape == (10_000, 768), f"LS.shape={LS.shape}"
assert ROW_INDICES.shape == (10_000,), f"ROW_INDICES.shape={ROW_INDICES.shape}"
assert LS.dtype == np.float64
print(f"LS.shape = {LS.shape}, ROW_INDICES[:5] = {ROW_INDICES[:5]}")

# --- fit_key reconstruction: the proof that ANALYSIS_CFG_BASE is Phase 1's cfg -------------
_cfg_incumbent = dict(ANALYSIS_CFG_BASE)
_cfg_incumbent["n_neighbors"] = K_INCUMBENT
_fit_key_check = config_key(_cfg_incumbent)
print(f"\nconfig_key(base | n_neighbors={K_INCUMBENT}) = {_fit_key_check}")
assert _fit_key_check == FIT_KEY_INCUMBENT, (
    f"reconstructed incumbent fit_key {_fit_key_check} != Phase 1's frozen "
    f"{FIT_KEY_INCUMBENT}. Some pinned field of ANALYSIS_CFG_BASE differs from the "
    f"configuration the k*=15 fit was made under, so the re-fits would not be comparable."
)
print(f"OK: reproduces Phase 1's frozen fit_key {FIT_KEY_INCUMBENT} exactly.")

FIT_KEYS = {}
for _k in K_ALL:
    _cfg = dict(ANALYSIS_CFG_BASE)
    _cfg["n_neighbors"] = _k
    FIT_KEYS[_k] = config_key(_cfg)
print("\nper-k fit_key (only n_neighbors differs):")
for _k in K_ALL:
    print(f"  k={_k:>2} -> {FIT_KEYS[_k]}")

cached subsample present before load: subsample_20260729_a79b3460b838fd0a.npz (117.4 MiB)


cached subsample byte-size and mtime unchanged after load: cache HIT confirmed (no re-download, no re-subsample).
LS.shape = (10000, 768), ROW_INDICES[:5] = [27 29 32 36 59]

config_key(base | n_neighbors=15) = 43cf438bc944c509
OK: reproduces Phase 1's frozen fit_key 43cf438bc944c509 exactly.

per-k fit_key (only n_neighbors differs):
  k= 5 -> 9db36086f7472619
  k=10 -> 9fbaf46e3570c8b7
  k=15 -> 43cf438bc944c509
  k=30 -> 860e4b66f08af831


## §4. Shared machinery

`_gate_classify` (Phase 2's classifier, 02-FINDINGS.md) reads its four thresholds from §2
above — no threshold literal inside the function — and is re-asserted against nine synthetic
boundary cases *before* it touches any real spectrum. `_draw_geo_pairs` mirrors
02-PATTERNS.md's `_draw_geo_pairs` idiom and is fed the same seed (`SEED + 2`), so the
200,000 point pairs used for `GEO_AMBIENT_RATIO` are **identical across all four k** —
pre-registration §4.4 requires exactly that. §5 cross-checks it: for k=15, the
`geo_pairs_r2` `_spectrum_arrays` samples and the one `_codiag_arrays` samples
independently must be bit-identical over the same fit.

In [4]:
def _gate_classify(r, m):
    """PASS/MARGINAL/FAIL classifier (02-FINDINGS.md's definition). Reads R_MAX_PASS,
    M_MAX_PASS, R_MAX_MARGINAL, M_MAX_MARGINAL from §2 -- no threshold literal appears in
    this function. PASS requires both r and m below their PASS thresholds; MARGINAL
    requires both below their (looser) MARGINAL thresholds; otherwise FAIL, every
    comparison strict less-than. Both conditions are conjunctions, so the returned verdict
    is already the worse of the two statistics."""
    if r < R_MAX_PASS and m < M_MAX_PASS:
        return "PASS"
    if r < R_MAX_MARGINAL and m < M_MAX_MARGINAL:
        return "MARGINAL"
    return "FAIL"


_synthetic_cases = [
    ((0.0, 0.0), "PASS"),
    ((0.099, 0.049), "PASS"),
    ((0.10, 0.0), "MARGINAL"),
    ((0.0, 0.05), "MARGINAL"),
    ((0.249, 0.149), "MARGINAL"),
    ((0.25, 0.0), "FAIL"),
    ((0.0, 0.15), "FAIL"),
    ((0.05, 0.20), "FAIL"),
]
print("=== §4: classifier boundary self-test (synthetic inputs, before any real data) ===")
print(f"{'r':>8} {'m':>8} {'expected':>10} {'actual':>10}")
for (_r_test, _m_test), _expected in _synthetic_cases:
    _actual = _gate_classify(_r_test, _m_test)
    print(f"{_r_test:>8.3f} {_m_test:>8.3f} {_expected:>10} {_actual:>10}")
    assert _actual == _expected, (
        f"_gate_classify({_r_test}, {_m_test}) = {_actual!r}, expected {_expected!r}"
    )
print("All classifier boundary cases passed (strict less-than at every boundary).")


def _draw_geo_pairs(rng, n_rows_total, count):
    """Draw `count` off-diagonal (row, col) index pairs, self-pairs rejected and redrawn
    (mirrors 02-PATTERNS.md's `_draw_geo_pairs` idiom, for pair-sample reproducibility)."""
    rows = rng.integers(0, n_rows_total, size=count)
    cols = rng.integers(0, n_rows_total, size=count)
    self_pairs = rows == cols
    while np.any(self_pairs):
        n_bad = int(self_pairs.sum())
        cols[self_pairs] = rng.integers(0, n_rows_total, size=n_bad)
        self_pairs = rows == cols
    return rows, cols


_r2_pair_rng = np.random.default_rng(R2_PAIR_SEED)
R2_PAIR_ROWS, R2_PAIR_COLS = _draw_geo_pairs(_r2_pair_rng, len(LS), R2_PAIR_COUNT)
assert R2_PAIR_ROWS.shape == (R2_PAIR_COUNT,)
assert R2_PAIR_COLS.shape == (R2_PAIR_COUNT,)
assert not np.any(R2_PAIR_ROWS == R2_PAIR_COLS), "self-pairs present in R2_PAIR_ROWS/COLS"
print(f"\nR2 pair sample drawn: {R2_PAIR_COUNT} pairs at seed {R2_PAIR_SEED} (= SEED+2)")
print("first three R2 pairs:",
      list(zip(R2_PAIR_ROWS[:3].tolist(), R2_PAIR_COLS[:3].tolist())))


def _peak_rss_mib():
    """Peak RSS in MiB (a monotone high-water mark), unit resolved by platform."""
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == "darwin":
        return raw / 1024**2, "bytes"
    return raw / 1024, "KiB"


def _current_rss_mib():
    """Current (not peak) resident set size in MiB, or NaN off Linux."""
    statm = Path("/proc/self/statm")
    if not statm.exists():
        return float("nan")
    return int(statm.read_text().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 1024**2


def _rss_line(label):
    peak, unit = _peak_rss_mib()
    return f"{label}: peak RSS {peak:.1f} MiB (ru_maxrss in {unit}) | current RSS {_current_rss_mib():.1f} MiB"


print()
print(_rss_line("after §4 setup"))

=== §4: classifier boundary self-test (synthetic inputs, before any real data) ===
       r        m   expected     actual
   0.000    0.000       PASS       PASS
   0.099    0.049       PASS       PASS
   0.100    0.000   MARGINAL   MARGINAL
   0.000    0.050   MARGINAL   MARGINAL
   0.249    0.149   MARGINAL   MARGINAL
   0.250    0.000       FAIL       FAIL
   0.000    0.150       FAIL       FAIL
   0.050    0.200       FAIL       FAIL
All classifier boundary cases passed (strict less-than at every boundary).

R2 pair sample drawn: 200000 pairs at seed 20260731 (= SEED+2)
first three R2 pairs: [(272, 970), (8884, 6454), (6434, 2808)]

after §4 setup: peak RSS 396.5 MiB (ru_maxrss in KiB) | current RSS 397.4 MiB


### §4.1 The two co-diagnostics (pre-registration §4.4)

    GEO_AMBIENT_RATIO(k) = median over the 200,000 pre-registered pairs of
                           ( graph geodesic distance / ambient Euclidean distance )

    LONG_EDGE_FRACTION(k) = fraction of kNN-graph edges whose length exceeds the
                            99th percentile of the k=15 graph's edge-length distribution

On a curved manifold the ratio exceeds 1 and grows with true separation; short-circuiting
collapses it toward 1. Both are **descriptive statistics with no pass/fail threshold** —
inventing one now, with the k=15 values already known, would be threshold-setting against a
seen result. They enter only through Rule B's qualitative test.

The ambient distance is Euclidean on the L2-normalized `LS` rows — the same metric Isomap
itself used, so the ratio's denominator is the same quantity its kNN graph is built from. It
is computed in 20,000-pair chunks: the full 200,000x768 difference array would be a 1.2 GiB
temporary for no reason.

In [5]:
AMBIENT_PAIRS = np.empty(R2_PAIR_COUNT, dtype=np.float64)
_chunk = 20_000
for _i0 in range(0, R2_PAIR_COUNT, _chunk):
    _i1 = min(_i0 + _chunk, R2_PAIR_COUNT)
    _diff = LS[R2_PAIR_ROWS[_i0:_i1]] - LS[R2_PAIR_COLS[_i0:_i1]]
    AMBIENT_PAIRS[_i0:_i1] = np.sqrt(np.einsum("ij,ij->i", _diff, _diff))
del _diff
gc.collect()

assert AMBIENT_PAIRS.shape == (R2_PAIR_COUNT,)
assert np.all(AMBIENT_PAIRS > 0), "zero ambient distance for a non-self pair"
print("=== §4.1: ambient Euclidean distances over the pre-registered pair sample ===")
print(f"count  = {AMBIENT_PAIRS.size}")
print(f"min    = {AMBIENT_PAIRS.min():.6f}")
print(f"median = {np.median(AMBIENT_PAIRS):.6f}")
print(f"max    = {AMBIENT_PAIRS.max():.6f}  (bounded by 2 on the unit sphere)")

=== §4.1: ambient Euclidean distances over the pre-registered pair sample ===
count  = 200000
min    = 0.135544
median = 0.569058
max    = 1.378718  (bounded by 2 on the unit sphere)


In [6]:
REFIT_COMPUTE = True  # cell-ordering anchor: must follow REFIT_PREREG (§2)

RESULTS = {}
LONG_EDGE_TAU = None  # set by §5 from the k=15 graph; the other k are measured against it


def _codiag_arrays(k, fit_key_k):
    """kNN-graph edge lengths and the geodesic pair sample for one k, read from the cached
    fit via mmap. Cached under its own npz so a re-run never re-reads the 1.66 GiB joblib."""
    cfg = {
        "fit_key": fit_key_k,
        "n_neighbors": k,
        "r2_pair_count": R2_PAIR_COUNT,
        "r2_pair_seed": R2_PAIR_SEED,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
    }

    def _compute():
        mapped = joblib.load(cache_path(f"isomap_{fit_key_k}", "joblib"), mmap_mode="r")
        dist_matrix = mapped.dist_matrix_
        assert dist_matrix.shape == (10_000, 10_000), f"{dist_matrix.shape}"
        assert dist_matrix.dtype == np.float64, f"{dist_matrix.dtype}"

        # Independent re-verification that THIS fit's own neighbour graph is connected
        # (T-01-08): a bridged graph must never be reported as connected.
        graph = mapped.nbrs_.kneighbors_graph(mode="distance")
        n_conn, _ = connected_components(graph, directed=False)
        edge_lengths = np.asarray(graph.data, dtype=np.float64)

        # Sampled HERE, while dist_matrix_ is mapped and indexable.
        geo = np.asarray(dist_matrix[R2_PAIR_ROWS, R2_PAIR_COLS], dtype=np.float64)
        assert np.all(geo > 0), "non-positive geodesic distance sampled"

        del mapped, dist_matrix, graph
        gc.collect()
        return {
            "edge_lengths": edge_lengths,
            "geo_pairs_r2": geo,
            "n_connected_components": np.array(n_conn),
        }

    return npz_cache(f"codiag_k{k}_{fit_key_k}", cfg, _compute)


def _spectrum_arrays(k, fit_key_k):
    """Full 10,000-value classical-MDS eigenspectrum for one k, computed by hand
    double-centring dist_matrix_ (§4 above)."""
    cfg = {
        "fit_key": fit_key_k,
        "n_neighbors": k,
        "d_sweep_max": D_SWEEP_MAX,
        "r2_pair_count": R2_PAIR_COUNT,
        "r2_pair_seed": R2_PAIR_SEED,
        "scipy_version": scipy.__version__,
        "numpy_version": np.__version__,
    }

    def _compute():
        t0 = time.perf_counter()
        mapped = joblib.load(cache_path(f"isomap_{fit_key_k}", "joblib"), mmap_mode="r")
        dist_matrix = mapped.dist_matrix_
        assert dist_matrix.shape == (10_000, 10_000)
        assert dist_matrix.dtype == np.float64

        # Sampled HERE, while dist_matrix_ is still mmap'd and indexable -- same draw
        # _codiag_arrays uses, kept as an independent copy for this artifact's own sidecar.
        geo_pairs_r2 = np.asarray(dist_matrix[R2_PAIR_ROWS, R2_PAIR_COLS], dtype=np.float64)
        assert np.all(geo_pairs_r2 > 0), "non-positive geodesic distance sampled"

        # np.array(..., copy=True) is REQUIRED rather than np.asarray: dist_matrix_ is a
        # read-only float64 memmap, and np.asarray with a matching dtype returns a VIEW of
        # that memmap (no copy), which makes the in-place **= 2 below raise
        # "output array is read-only".
        D2 = np.array(dist_matrix, dtype=np.float64, copy=True)
        del mapped, dist_matrix
        gc.collect()
        D2 **= 2

        # Chunk-wise symmetry check, avoiding a second 800 MB temporary. The symmetric
        # eigensolver reads a single triangle -- an asymmetric input would be silently and
        # wrongly reduced rather than rejected.
        n = D2.shape[0]
        max_asym = 0.0
        for i0 in range(0, n, 1_000):
            i1 = min(i0 + 1_000, n)
            block_max = float(np.abs(D2[i0:i1, :] - D2[:, i0:i1].T).max())
            max_asym = max(max_asym, block_max)
        bound = SYMMETRY_RTOL * float(D2.max())
        assert max_asym <= bound, (
            f"D2 not symmetric within SYMMETRY_RTOL: {max_asym:.3e} > {bound:.3e}"
        )
        print(f"    symmetry OK: max |D2 - D2.T| = {max_asym:.3e} <= bound {bound:.3e}")

        # In-place mean-form double-centring (D-09).
        row_mean = D2.mean(axis=1, keepdims=True)
        col_mean = D2.mean(axis=0, keepdims=True)
        grand_mean = D2.mean()
        D2 -= row_mean
        D2 -= col_mean
        D2 += grand_mean
        D2 *= -0.5
        B = D2
        print(f"    {_rss_line('    after in-place double-centring')}")

        # Split eigensolve (D-10): all 10,000 values, then the top eigenpairs only -- no
        # 10,000x10,000 eigenvector array is ever materialised.
        eigvals_all = scipy.linalg.eigvalsh(B)
        n_positive = int(np.sum(eigvals_all > 0))
        k_eff = min(D_SWEEP_MAX, n_positive)
        eigvals_top_asc, eigvecs_top_asc = scipy.linalg.eigh(
            B, subset_by_index=[n - k_eff, n - 1]
        )
        eigvals_top = eigvals_top_asc[::-1]
        eigvecs_top = eigvecs_top_asc[:, ::-1]  # descending order, matching eigvals_top
        del eigvecs_top_asc, eigvals_top_asc, B, D2
        gc.collect()

        spectrum_seconds = time.perf_counter() - t0
        return {
            "eigvals_all": eigvals_all.astype(np.float64),
            "eigvals_top": eigvals_top.astype(np.float64),
            "eigvecs_top": eigvecs_top.astype(np.float64),
            "geo_pairs_r2": geo_pairs_r2,
            "n_positive": np.array(n_positive),
            "k_eff": np.array(k_eff),
            "max_asym": np.array(max_asym),
            "spectrum_seconds": np.array(spectrum_seconds),
        }

    return npz_cache(f"mds_eigenspectrum_{fit_key_k}", cfg, _compute)


def _gate_stats(eigvals_all):
    """r, m and the verdict, computed over the whole spectrum, no zero-thresholding, no
    filtering of the near-zero tail."""
    negative_mask = eigvals_all < 0
    lambda_max_pos = float(eigvals_all[-1])
    if negative_mask.any():
        lambda_min_neg = float(eigvals_all[0])
        r = abs(lambda_min_neg) / lambda_max_pos
        m = float(np.abs(eigvals_all[negative_mask]).sum() / np.abs(eigvals_all).sum())
    else:
        lambda_min_neg = 0.0
        r = 0.0
        m = 0.0
    return {
        "r": r,
        "m": m,
        "verdict": _gate_classify(r, m),
        "lambda_max_pos": lambda_max_pos,
        "lambda_min_neg": lambda_min_neg,
        "n_negative": int(negative_mask.sum()),
        "noise_floor": float(10_000 * np.finfo(np.float64).eps * lambda_max_pos),
    }


def _process_k(k):
    """Fit-or-load, spectrum, gate statistics and both co-diagnostics for a single k.
    Exactly one k is held in memory at a time; every large binding is released before
    returning."""
    print("=" * 78)
    print(f"### k = {k}")
    print(_rss_line(f"before k={k}"))
    fit_key_k = FIT_KEYS[k]
    print(f"fit_key = {fit_key_k}")

    fit_cfg = dict(ANALYSIS_CFG_BASE)
    fit_cfg["n_neighbors"] = k
    assert config_key(fit_cfg) == fit_key_k

    fit_seconds = {}

    def _fit():
        # eigen_solver="dense" is pinned explicitly, not left at "auto": Isomap exposes no
        # random_state and the "auto"/"arpack" paths use ARPACK's Lanczos iteration with a
        # random start vector, so "dense" is what makes this fit deterministic LAPACK (D-15).
        t0 = time.perf_counter()
        model = Isomap(
            n_neighbors=k,
            n_components=N_COMPONENTS,
            eigen_solver="dense",
            n_jobs=-1,
            path_method="auto",
        )
        model.fit(LS)
        fit_seconds["s"] = time.perf_counter() - t0
        return model

    t_fit = time.perf_counter()
    model = joblib_cache(f"isomap_{fit_key_k}", fit_cfg, _fit)
    assert model.dist_matrix_.shape == (10_000, 10_000), f"{model.dist_matrix_.shape}"
    assert model.dist_matrix_.dtype == np.float64
    assert model.embedding_.shape == (10_000, N_COMPONENTS), f"{model.embedding_.shape}"
    # Captured before release so the top-18 cross-check works on a warm re-run too.
    eig_sklearn = np.asarray(model.kernel_pca_.eigenvalues_, dtype=np.float64)
    assert eig_sklearn.shape == (N_COMPONENTS,)
    del model
    gc.collect()
    if "s" in fit_seconds:
        print(f"  fit: FRESH, {fit_seconds['s']:.1f}s")
    else:
        print(f"  fit: cache hit, loaded in {time.perf_counter() - t_fit:.1f}s")
    joblib_mib = cache_path(f"isomap_{fit_key_k}", "joblib").stat().st_size / 1024**2
    print(f"  isomap_{fit_key_k}.joblib = {joblib_mib:.1f} MiB")

    # --- spectrum -------------------------------------------------------------------
    t_spec = time.perf_counter()
    spec = _spectrum_arrays(k, fit_key_k)
    print(f"  spectrum: {time.perf_counter() - t_spec:.1f}s "
          f"(spectrum_seconds={float(spec['spectrum_seconds']):.1f})")

    eigvals_all = np.asarray(spec["eigvals_all"], dtype=np.float64)
    assert eigvals_all.shape == (10_000,), (
        f"eigvals_all.shape={eigvals_all.shape}, expected (10000,) -- this length is "
        f"exactly what a truncated attribute (kernel_pca_.eigenvalues_, 18 values) cannot "
        f"produce. It is the mechanical proof that the audited spectrum is whole."
    )
    assert eigvals_all.dtype == np.float64, f"eigvals_all.dtype={eigvals_all.dtype}"
    assert np.all(np.diff(eigvals_all) >= -1e-9), "eigvals_all is not ascending"
    n_positive = int(spec["n_positive"])
    eigvals_top = np.asarray(spec["eigvals_top"], dtype=np.float64)

    # Cross-check of the leading 18 values only against sklearn's own truncated
    # eigenvalues. Never the source of the audited spectrum -- it has 18 entries.
    worst_rel = float(
        np.max(np.abs(eigvals_top[:18] - eig_sklearn) / np.abs(eig_sklearn))
    )
    assert np.allclose(eigvals_top[:18], eig_sklearn, rtol=1e-8), (
        f"hand-rolled top-18 disagree with sklearn's kernel_pca_.eigenvalues_ "
        f"(worst rel diff {worst_rel:.3e})"
    )
    print(f"  sklearn top-18 cross-check OK (rtol=1e-8, worst rel diff {worst_rel:.3e})")

    stats = _gate_stats(eigvals_all)

    # --- co-diagnostics -------------------------------------------------------------
    codiag = _codiag_arrays(k, fit_key_k)
    n_conn = int(codiag["n_connected_components"])
    assert n_conn == 1, (
        f"k={k}: independently re-verified n_connected_components={n_conn}, expected 1"
    )
    edge_lengths = np.asarray(codiag["edge_lengths"], dtype=np.float64)
    geo_pairs = np.asarray(codiag["geo_pairs_r2"], dtype=np.float64)
    assert edge_lengths.shape == (10_000 * k,), f"{edge_lengths.shape} for k={k}"
    assert geo_pairs.shape == (R2_PAIR_COUNT,)

    geo_ambient_ratio = float(np.median(geo_pairs / AMBIENT_PAIRS))

    record = {
        "k": k,
        "fit_key": fit_key_k,
        "n_connected_components": n_conn,
        "n_positive": n_positive,
        "n_negative": stats["n_negative"],
        "r": stats["r"],
        "m": stats["m"],
        "verdict": stats["verdict"],
        "lambda_max_pos": stats["lambda_max_pos"],
        "lambda_min_neg": stats["lambda_min_neg"],
        "noise_floor": stats["noise_floor"],
        "geo_ambient_ratio": geo_ambient_ratio,
        "geo_ambient_p90": float(np.percentile(geo_pairs / AMBIENT_PAIRS, 90)),
        "edge_length_mean": float(edge_lengths.mean()),
        "edge_length_p99": float(np.percentile(edge_lengths, 99)),
        "edge_length_max": float(edge_lengths.max()),
        "n_edges": int(edge_lengths.size),
        "geo_median": float(np.median(geo_pairs)),
    }

    print(f"  n_positive = {n_positive}, n_negative = {stats['n_negative']}")
    print(f"  lambda_max_pos = {stats['lambda_max_pos']:.6e}, "
          f"lambda_min_neg = {stats['lambda_min_neg']:.6e}")
    print(f"  noise floor    = {stats['noise_floor']:.3e} -- "
          f"|lambda_min_neg| is "
          f"{'ABOVE' if abs(stats['lambda_min_neg']) > stats['noise_floor'] else 'at/below'} it")
    print(f"  r = {stats['r']:.6f}  (R_MAX_PASS={R_MAX_PASS}, R_MAX_MARGINAL={R_MAX_MARGINAL})")
    print(f"  m = {stats['m']:.6f}  (M_MAX_PASS={M_MAX_PASS}, M_MAX_MARGINAL={M_MAX_MARGINAL})")
    print(f"  VERDICT = {stats['verdict']}")
    print(f"  GEO_AMBIENT_RATIO = {geo_ambient_ratio:.6f}  "
          f"(median geodesic/ambient over the {R2_PAIR_COUNT} pre-registered pairs)")

    del eigvals_all, eigvals_top, spec, codiag, edge_lengths, geo_pairs, eig_sklearn
    gc.collect()
    print(_rss_line(f"after k={k}"))
    return record

In [7]:
# Cell-ordering self-assertion (the same device 01 §6.1 uses): the pre-registered constants
# must execute before the machinery that consumes them. This notebook reads its own JSON.
_nb_path = NOTEBOOK_DIR / "02_k_sensitivity_refit.ipynb"
_nb_json = _json.loads(_nb_path.read_text())
_src = ["".join(c["source"]) for c in _nb_json["cells"] if c["cell_type"] == "code"]
_prereg_idx = next(i for i, s in enumerate(_src) if "REFIT_PREREG" in s and "R_MAX_PASS" in s)
_compute_idx = next(i for i, s in enumerate(_src) if "REFIT_COMPUTE" in s and "mmap_mode" in s)
assert _prereg_idx < _compute_idx, (
    f"PRE-REGISTRATION VIOLATED: REFIT_PREREG cell (code-cell index {_prereg_idx}) must "
    f"precede the REFIT_COMPUTE cell (code-cell index {_compute_idx})."
)
print(f"Pre-registration ordering OK: REFIT_PREREG code-cell index {_prereg_idx} < "
      f"REFIT_COMPUTE code-cell index {_compute_idx}")

Pre-registration ordering OK: REFIT_PREREG code-cell index 1 < REFIT_COMPUTE code-cell index 5


## §5. Baseline: the incumbent k=15

k=15 is fit and its spectrum computed by this notebook on the same `_process_k` path as
every other k (§4) -- it is no longer read back from a separate artifact.

Two things are established here that the rest of the analysis depends on:

1. **`LONG_EDGE_TAU`** -- the 99th percentile of the k=15 kNN-graph edge-length distribution,
   the fixed reference every `LONG_EDGE_FRACTION(k)` is measured against. `LONG_EDGE_FRACTION(15)`
   is therefore ~0.01 by construction; that is a sanity check on the definition, not a result.
2. **A cross-path check on the pair sample** -- the geodesic distances `_spectrum_arrays`
   samples while computing k=15's spectrum are asserted bit-identical to the ones
   `_codiag_arrays` samples independently for the same fit.

`r` and `m` are also regression-checked against the published `0.052419` / `0.412071`.

In [ ]:
RESULTS[K_INCUMBENT] = _process_k(K_INCUMBENT)

# --- Regression check against 02-01-SUMMARY.md's published statistics ---------------------
assert abs(RESULTS[K_INCUMBENT]["r"] - R_STAT_INCUMBENT_PUBLISHED) < 5e-7, (
    f"r(15)={RESULTS[K_INCUMBENT]['r']!r} does not reproduce the published "
    f"{R_STAT_INCUMBENT_PUBLISHED}"
)
assert abs(RESULTS[K_INCUMBENT]["m"] - M_STAT_INCUMBENT_PUBLISHED) < 5e-7, (
    f"m(15)={RESULTS[K_INCUMBENT]['m']!r} does not reproduce the published "
    f"{M_STAT_INCUMBENT_PUBLISHED}"
)
print(f"\nRegression check OK: r(15)={RESULTS[K_INCUMBENT]['r']:.6f}, "
      f"m(15)={RESULTS[K_INCUMBENT]['m']:.6f} reproduce 02-01-SUMMARY.md exactly.")

# --- Cross-path check: the pair sample _spectrum_arrays drew matches _codiag_arrays's draw -
_spec_15 = _spectrum_arrays(K_INCUMBENT, FIT_KEYS[K_INCUMBENT])
_geo_spectrum = np.asarray(_spec_15["geo_pairs_r2"], dtype=np.float64)
_codiag_15 = _codiag_arrays(K_INCUMBENT, FIT_KEYS[K_INCUMBENT])
_geo_codiag = np.asarray(_codiag_15["geo_pairs_r2"], dtype=np.float64)
assert np.array_equal(_geo_spectrum, _geo_codiag), (
    "geo_pairs_r2 from _spectrum_arrays and _codiag_arrays diverge for k=15 -- the two "
    "code paths must reach the same pair sample over the same fit."
)
print(f"Pair-sample cross-path check OK: the {R2_PAIR_COUNT} geodesic distances agree "
      f"between _spectrum_arrays and _codiag_arrays for k=15.")

# --- LONG_EDGE_TAU: the fixed k=15 reference for LONG_EDGE_FRACTION ------------------------
_edges_15 = np.asarray(_codiag_15["edge_lengths"], dtype=np.float64)
LONG_EDGE_TAU = float(np.percentile(_edges_15, 99))
RESULTS[K_INCUMBENT]["long_edge_fraction"] = float(np.mean(_edges_15 > LONG_EDGE_TAU))

print(f"\n=== §5: LONG_EDGE_TAU (99th percentile of the k=15 graph's edge lengths) ===")
print(f"k=15 edge count      = {_edges_15.size} (= 10,000 x 15)")
print(f"k=15 edge length min/median/max = {_edges_15.min():.6f} / "
      f"{np.median(_edges_15):.6f} / {_edges_15.max():.6f}")
print(f"LONG_EDGE_TAU        = {LONG_EDGE_TAU:.6f}")
print(f"LONG_EDGE_FRACTION(15) = {RESULTS[K_INCUMBENT]['long_edge_fraction']:.6f} "
      f"(~0.01 by construction -- a definition check, not a result)")
print(f"GEO_AMBIENT_RATIO(15)  = {RESULTS[K_INCUMBENT]['geo_ambient_ratio']:.6f}")

del _edges_15, _geo_spectrum, _geo_codiag, _codiag_15, _spec_15
gc.collect()
print()
print(_rss_line("after §5"))

## §6. Re-fit at k = 5

Fresh Isomap fit and spectrum via the shared §4 machinery (`_process_k`); `r`/`m`/the
verdict come from the same `_gate_classify` and the same four thresholds.

In [9]:
RESULTS[5] = _process_k(5)
RESULTS[5]["long_edge_fraction"] = None  # filled below, once LONG_EDGE_TAU is in scope

_codiag = _codiag_arrays(5, FIT_KEYS[5])
_edges = np.asarray(_codiag["edge_lengths"], dtype=np.float64)
RESULTS[5]["long_edge_fraction"] = float(np.mean(_edges > LONG_EDGE_TAU))
print(f"\nLONG_EDGE_FRACTION(5) = {RESULTS[5]['long_edge_fraction']:.6f} "
      f"(edges longer than LONG_EDGE_TAU={LONG_EDGE_TAU:.6f}, the k=15 99th percentile)")
print(f"GEO_AMBIENT_RATIO(5)  = {RESULTS[5]['geo_ambient_ratio']:.6f}  "
      f"vs k=15 baseline {RESULTS[K_INCUMBENT]['geo_ambient_ratio']:.6f}")
del _edges, _codiag
gc.collect()
print(_rss_line("after k=5 co-diagnostics"))

### k = 5
before k=5: peak RSS 1986.8 MiB (ru_maxrss in KiB) | current RSS 400.4 MiB
fit_key = 9db36086f7472619


  fit: cache hit, loaded in 1.1s
  isomap_9db36086f7472619.joblib = 1587.3 MiB
  spectrum: 0.0s (spectrum_seconds=122.9)
  sklearn top-18 cross-check OK (rtol=1e-8, worst rel diff 4.393e-15)
  n_positive = 4972, n_negative = 5028
  lambda_max_pos = 5.432086e+03, lambda_min_neg = -3.276213e+02
  noise floor    = 1.206e-08 -- |lambda_min_neg| is ABOVE it
  r = 0.060312  (R_MAX_PASS=0.1, R_MAX_MARGINAL=0.25)
  m = 0.406433  (M_MAX_PASS=0.05, M_MAX_MARGINAL=0.15)
  VERDICT = FAIL
  GEO_AMBIENT_RATIO = 2.828727  (median geodesic/ambient over the 200000 pre-registered pairs)
after k=5: peak RSS 1986.8 MiB (ru_maxrss in KiB) | current RSS 405.3 MiB

LONG_EDGE_FRACTION(5) = 0.006540 (edges longer than LONG_EDGE_TAU=0.516666, the k=15 99th percentile)
GEO_AMBIENT_RATIO(5)  = 2.828727  vs k=15 baseline 2.117401


after k=5 co-diagnostics: peak RSS 1986.8 MiB (ru_maxrss in KiB) | current RSS 405.3 MiB


## §7. Re-fit at k = 10

Fresh Isomap fit and spectrum via the shared §4 machinery (`_process_k`); `r`/`m`/the
verdict come from the same `_gate_classify` and the same four thresholds.

In [10]:
RESULTS[10] = _process_k(10)
RESULTS[10]["long_edge_fraction"] = None  # filled below, once LONG_EDGE_TAU is in scope

_codiag = _codiag_arrays(10, FIT_KEYS[10])
_edges = np.asarray(_codiag["edge_lengths"], dtype=np.float64)
RESULTS[10]["long_edge_fraction"] = float(np.mean(_edges > LONG_EDGE_TAU))
print(f"\nLONG_EDGE_FRACTION(10) = {RESULTS[10]['long_edge_fraction']:.6f} "
      f"(edges longer than LONG_EDGE_TAU={LONG_EDGE_TAU:.6f}, the k=15 99th percentile)")
print(f"GEO_AMBIENT_RATIO(10)  = {RESULTS[10]['geo_ambient_ratio']:.6f}  "
      f"vs k=15 baseline {RESULTS[K_INCUMBENT]['geo_ambient_ratio']:.6f}")
del _edges, _codiag
gc.collect()
print(_rss_line("after k=10 co-diagnostics"))

### k = 10
before k=10: peak RSS 1986.8 MiB (ru_maxrss in KiB) | current RSS 405.3 MiB
fit_key = 9fbaf46e3570c8b7


  fit: cache hit, loaded in 1.1s
  isomap_9fbaf46e3570c8b7.joblib = 1587.3 MiB
  spectrum: 0.0s (spectrum_seconds=120.4)
  sklearn top-18 cross-check OK (rtol=1e-8, worst rel diff 5.613e-15)
  n_positive = 4971, n_negative = 5029
  lambda_max_pos = 3.798254e+03, lambda_min_neg = -2.214809e+02
  noise floor    = 8.434e-09 -- |lambda_min_neg| is ABOVE it
  r = 0.058311  (R_MAX_PASS=0.1, R_MAX_MARGINAL=0.25)
  m = 0.410187  (M_MAX_PASS=0.05, M_MAX_MARGINAL=0.15)
  VERDICT = FAIL
  GEO_AMBIENT_RATIO = 2.320592  (median geodesic/ambient over the 200000 pre-registered pairs)
after k=10: peak RSS 1988.8 MiB (ru_maxrss in KiB) | current RSS 405.7 MiB

LONG_EDGE_FRACTION(10) = 0.008620 (edges longer than LONG_EDGE_TAU=0.516666, the k=15 99th percentile)
GEO_AMBIENT_RATIO(10)  = 2.320592  vs k=15 baseline 2.117401
after k=10 co-diagnostics: peak RSS 1988.8 MiB (ru_maxrss in KiB) | current RSS 405.7 MiB


## §8. Re-fit at k = 30

Fresh Isomap fit and spectrum via the shared §4 machinery (`_process_k`); `r`/`m`/the
verdict come from the same `_gate_classify` and the same four thresholds.

k=30 is the densest graph in the pre-registered set and therefore the one where the §3
confound bites hardest: under H2 it should show the lowest `m`, but it is also the k most
likely to have short-circuited the manifold. That is precisely what the two co-diagnostics
exist to separate.

In [11]:
RESULTS[30] = _process_k(30)
RESULTS[30]["long_edge_fraction"] = None  # filled below, once LONG_EDGE_TAU is in scope

_codiag = _codiag_arrays(30, FIT_KEYS[30])
_edges = np.asarray(_codiag["edge_lengths"], dtype=np.float64)
RESULTS[30]["long_edge_fraction"] = float(np.mean(_edges > LONG_EDGE_TAU))
print(f"\nLONG_EDGE_FRACTION(30) = {RESULTS[30]['long_edge_fraction']:.6f} "
      f"(edges longer than LONG_EDGE_TAU={LONG_EDGE_TAU:.6f}, the k=15 99th percentile)")
print(f"GEO_AMBIENT_RATIO(30)  = {RESULTS[30]['geo_ambient_ratio']:.6f}  "
      f"vs k=15 baseline {RESULTS[K_INCUMBENT]['geo_ambient_ratio']:.6f}")
del _edges, _codiag
gc.collect()
print(_rss_line("after k=30 co-diagnostics"))

### k = 30
before k=30: peak RSS 1988.8 MiB (ru_maxrss in KiB) | current RSS 405.7 MiB
fit_key = 860e4b66f08af831


  fit: cache hit, loaded in 1.1s
  isomap_860e4b66f08af831.joblib = 1587.3 MiB
  spectrum: 0.0s (spectrum_seconds=122.8)
  sklearn top-18 cross-check OK (rtol=1e-8, worst rel diff 5.341e-15)
  n_positive = 4963, n_negative = 5037
  lambda_max_pos = 2.528065e+03, lambda_min_neg = -1.281927e+02
  noise floor    = 5.613e-09 -- |lambda_min_neg| is ABOVE it
  r = 0.050708  (R_MAX_PASS=0.1, R_MAX_MARGINAL=0.25)
  m = 0.415735  (M_MAX_PASS=0.05, M_MAX_MARGINAL=0.15)
  VERDICT = FAIL
  GEO_AMBIENT_RATIO = 1.864727  (median geodesic/ambient over the 200000 pre-registered pairs)
after k=30: peak RSS 1989.0 MiB (ru_maxrss in KiB) | current RSS 400.5 MiB

LONG_EDGE_FRACTION(30) = 0.013923 (edges longer than LONG_EDGE_TAU=0.516666, the k=15 99th percentile)
GEO_AMBIENT_RATIO(30)  = 1.864727  vs k=15 baseline 2.117401


after k=30 co-diagnostics: peak RSS 1989.0 MiB (ru_maxrss in KiB) | current RSS 404.9 MiB


## §9. The comparison table

Reported for all four k regardless of outcome (pre-registration §6). No k is dropped, and
the table is not sorted by verdict.

In [12]:
print("=" * 118)
print("k-SENSITIVITY COMPARISON TABLE  (thresholds: r < 0.10 / 0.25, m < 0.05 / 0.15, strict)")
print("=" * 118)
_hdr = (f"{'k':>4} {'r(k)':>10} {'m(k)':>10} {'n_positive':>11} {'n_negative':>11} "
        f"{'GEO_AMB_RATIO':>14} {'LONG_EDGE_FRAC':>15} {'verdict':>9}")
print(_hdr)
print("-" * len(_hdr))
for _k in K_ALL:
    _r = RESULTS[_k]
    _tag = "  <- incumbent" if _k == K_INCUMBENT else ""
    print(f"{_k:>4} {_r['r']:>10.6f} {_r['m']:>10.6f} {_r['n_positive']:>11} "
          f"{_r['n_negative']:>11} {_r['geo_ambient_ratio']:>14.6f} "
          f"{_r['long_edge_fraction']:>15.6f} {_r['verdict']:>9}{_tag}")
print("-" * len(_hdr))

print("\nSupporting detail (descriptive; no thresholds attached):")
_hdr2 = (f"{'k':>4} {'lambda_max_pos':>15} {'lambda_min_neg':>15} {'noise_floor':>12} "
         f"{'n_edges':>9} {'edge_p99':>10} {'edge_max':>10} {'median_geo':>11}")
print(_hdr2)
print("-" * len(_hdr2))
for _k in K_ALL:
    _r = RESULTS[_k]
    print(f"{_k:>4} {_r['lambda_max_pos']:>15.6e} {_r['lambda_min_neg']:>15.6e} "
          f"{_r['noise_floor']:>12.3e} {_r['n_edges']:>9} {_r['edge_length_p99']:>10.6f} "
          f"{_r['edge_length_max']:>10.6f} {_r['geo_median']:>11.6f}")
print("-" * len(_hdr2))

for _k in K_ALL:
    assert RESULTS[_k]["n_positive"] + RESULTS[_k]["n_negative"] <= 10_000
    assert RESULTS[_k]["n_connected_components"] == 1
print("\nAll four graphs independently re-verified connected (n_components == 1).")
print(f"LONG_EDGE_TAU = {LONG_EDGE_TAU:.6f} (99th percentile of the k=15 edge lengths); "
      f"LONG_EDGE_FRACTION(15) = {RESULTS[K_INCUMBENT]['long_edge_fraction']:.6f} "
      f"is ~0.01 by construction.")

k-SENSITIVITY COMPARISON TABLE  (thresholds: r < 0.10 / 0.25, m < 0.05 / 0.15, strict)
   k       r(k)       m(k)  n_positive  n_negative  GEO_AMB_RATIO  LONG_EDGE_FRAC   verdict
-------------------------------------------------------------------------------------------
   5   0.060312   0.406433        4972        5028       2.828727        0.006540      FAIL
  10   0.058311   0.410187        4971        5029       2.320592        0.008620      FAIL
  15   0.052419   0.412071        4971        5029       2.117401        0.010000      FAIL  <- incumbent
  30   0.050708   0.415735        4963        5037       1.864727        0.013923      FAIL
-------------------------------------------------------------------------------------------

Supporting detail (descriptive; no thresholds attached):
   k  lambda_max_pos  lambda_min_neg  noise_floor   n_edges   edge_p99   edge_max  median_geo
---------------------------------------------------------------------------------------------
   5    5

## §10. Applying the pre-registered interpretation rule (§5, Rules A-D)

The rule was fixed before any fit ran. It is applied mechanically here — the code below
tests the conditions in the order the pre-registration states them and prints which rule
fired and why. It has no branch that adopts a new `k*`: Rule C explicitly forbids this
analysis from doing that.

- **Rule A** — `m(k) >= M_MAX_MARGINAL` for all k in {5, 10, 15, 30}: the non-Euclideanity is
  not an artifact of neighbourhood scale, H2 is not supported, and `GATE_VERDICT = FAIL`
  stands as the Phase 2 outcome.
- **Rule B** — some k has `m(k) < M_MAX_MARGINAL` **and** `r(k) < R_MAX_MARGINAL`: that k is a
  *candidate only* and must then pass the qualitative short-circuit test
  (`GEO_AMBIENT_RATIO` must not fall materially below the k=15 value, and
  `LONG_EDGE_FRACTION` must not rise materially above it). A candidate that fails is
  rejected *with its numbers recorded*, and Rule A's outcome applies.
- **Rule C** — a candidate survives the short-circuit test: it is *reported as a finding*.
  Adopting it requires re-running Phase 1's stage-2 stability selection and a separate
  documented amendment to Phase 1.
- **Rule D** — `m(k)` decreases monotonically with k but no k clears `M_MAX_MARGINAL`: report
  the trend as evidence bearing on H1 vs H2 and apply Rule A. A trend in the predicted
  direction is not a PASS.

In [13]:
_m = {k: RESULTS[k]["m"] for k in K_ALL}
_r = {k: RESULTS[k]["r"] for k in K_ALL}

print("=== §10: applying the pre-registered interpretation rule ===\n")
print("Rule B test -- does any k satisfy BOTH m(k) < M_MAX_MARGINAL and r(k) < R_MAX_MARGINAL?")
print(f"{'k':>4} {'m(k)':>10} {'< 0.15':>8} {'r(k)':>10} {'< 0.25':>8} {'candidate':>10}")
CANDIDATES = []
for _k in K_ALL:
    _m_ok = _m[_k] < M_MAX_MARGINAL
    _r_ok = _r[_k] < R_MAX_MARGINAL
    _cand = _m_ok and _r_ok
    if _cand:
        CANDIDATES.append(_k)
    print(f"{_k:>4} {_m[_k]:>10.6f} {str(_m_ok):>8} {_r[_k]:>10.6f} {str(_r_ok):>8} "
          f"{str(_cand):>10}")
print(f"\nCANDIDATES = {CANDIDATES}")

_m_ordered = [_m[k] for k in sorted(K_ALL)]
MONOTONE_DECREASING = all(b < a for a, b in zip(_m_ordered, _m_ordered[1:]))
RULE_A_HOLDS = all(_m[k] >= M_MAX_MARGINAL for k in K_ALL)
print(f"m(k) in ascending k order: "
      + ", ".join(f"m({k})={_m[k]:.6f}" for k in sorted(K_ALL)))
print(f"strictly monotone decreasing in k? {MONOTONE_DECREASING}")
print(f"Rule A precondition (m(k) >= {M_MAX_MARGINAL} for ALL k)? {RULE_A_HOLDS}")

if CANDIDATES:
    RULE_FIRED = "B"
    print("\n>>> Rule B applies: at least one k clears both MARGINAL bounds. Each candidate "
          "is a CANDIDATE ONLY, never an adopted k*, and must pass the short-circuit test.")
    _base_ratio = RESULTS[K_INCUMBENT]["geo_ambient_ratio"]
    _base_lef = RESULTS[K_INCUMBENT]["long_edge_fraction"]
    SURVIVORS = []
    for _k in CANDIDATES:
        _ratio = RESULTS[_k]["geo_ambient_ratio"]
        _lef = RESULTS[_k]["long_edge_fraction"]
        print(f"\n  candidate k={_k}: GEO_AMBIENT_RATIO {_ratio:.6f} vs k=15 {_base_ratio:.6f} "
              f"({100 * (_ratio - _base_ratio) / _base_ratio:+.2f}%); "
              f"LONG_EDGE_FRACTION {_lef:.6f} vs k=15 {_base_lef:.6f} "
              f"({100 * (_lef - _base_lef) / _base_lef:+.2f}%)")
        print("  The short-circuit test is QUALITATIVE by pre-registration §4.4 -- the two "
              "co-diagnostics carry no pass/fail threshold, and inventing one now would be "
              "threshold-setting against a seen result. The numbers above are recorded for "
              "the human judgement Rule B calls for.")
        SURVIVORS.append(_k)
    if SURVIVORS:
        RULE_FIRED = "B (short-circuit test reported; Rule C reporting-only posture applies "
        RULE_FIRED += "to any survivor -- no k* is adopted here)"
elif MONOTONE_DECREASING and RULE_A_HOLDS:
    RULE_FIRED = "D"
    print("\n>>> Rule D applies: m(k) decreases monotonically with k but NO k clears "
          f"M_MAX_MARGINAL={M_MAX_MARGINAL}. The trend is reported as evidence bearing on "
          "H1 vs H2, and Rule A's outcome applies. A trend in the predicted direction is "
          "not a PASS.")
elif RULE_A_HOLDS:
    RULE_FIRED = "A"
    print(f"\n>>> Rule A applies: m(k) >= M_MAX_MARGINAL={M_MAX_MARGINAL} for every k in "
          f"{K_ALL}, and the m(k) sequence is not monotone decreasing in k. The "
          "non-Euclideanity is not an artifact of neighbourhood scale; H2 is not supported.")
else:
    RULE_FIRED = "NONE"
    print("\n>>> No pre-registered rule matched cleanly (some k has m < M_MAX_MARGINAL but "
          "r >= R_MAX_MARGINAL, so it is neither a Rule B candidate nor a Rule A "
          "counterexample). This is reported as-is rather than resolved by an unregistered "
          "decision.")

print(f"\nRULE_FIRED = {RULE_FIRED}")
print(f"GATE_VERDICT for the incumbent k*=15 fit stands as: {RESULTS[K_INCUMBENT]['verdict']}")
print("No k* is adopted by this analysis (pre-registration Rule C).")

=== §10: applying the pre-registered interpretation rule ===

Rule B test -- does any k satisfy BOTH m(k) < M_MAX_MARGINAL and r(k) < R_MAX_MARGINAL?
   k       m(k)   < 0.15       r(k)   < 0.25  candidate
   5   0.406433    False   0.060312     True      False
  10   0.410187    False   0.058311     True      False
  15   0.412071    False   0.052419     True      False
  30   0.415735    False   0.050708     True      False

CANDIDATES = []
m(k) in ascending k order: m(5)=0.406433, m(10)=0.410187, m(15)=0.412071, m(30)=0.415735
strictly monotone decreasing in k? False
Rule A precondition (m(k) >= 0.15 for ALL k)? True

>>> Rule A applies: m(k) >= M_MAX_MARGINAL=0.15 for every k in [5, 10, 15, 30], and the m(k) sequence is not monotone decreasing in k. The non-Euclideanity is not an artifact of neighbourhood scale; H2 is not supported.

RULE_FIRED = A
GATE_VERDICT for the incumbent k*=15 fit stands as: FAIL
No k* is adopted by this analysis (pre-registration Rule C).


### §10.1 Persisted artifact

The whole table plus the fired rule is written to `notebooks/.cache/` through the same
`json_cache` helper the rest of the milestone uses, so the numbers appended to the
pre-registration's §8 Outcome have a machine-readable source that is not this notebook's
printed output.

In [14]:
_table_cfg = {
    "k_refit": K_REFIT,
    "k_incumbent": K_INCUMBENT,
    "r_max_pass": R_MAX_PASS,
    "m_max_pass": M_MAX_PASS,
    "r_max_marginal": R_MAX_MARGINAL,
    "m_max_marginal": M_MAX_MARGINAL,
    "r2_pair_count": R2_PAIR_COUNT,
    "r2_pair_seed": R2_PAIR_SEED,
    "fit_keys": {str(k): FIT_KEYS[k] for k in K_ALL},
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}


def _build_table():
    return {
        "rule_fired": RULE_FIRED,
        "candidates": CANDIDATES,
        "monotone_decreasing_m": bool(MONOTONE_DECREASING),
        "rule_a_precondition_holds": bool(RULE_A_HOLDS),
        "long_edge_tau": LONG_EDGE_TAU,
        "incumbent_verdict": RESULTS[K_INCUMBENT]["verdict"],
        "rows": [RESULTS[k] for k in K_ALL],
    }


K_SENSITIVITY_TABLE = json_cache(
    f"k_sensitivity_refit_{FIT_KEY_INCUMBENT}", _table_cfg, _build_table
)
print(_json.dumps(K_SENSITIVITY_TABLE, indent=2, sort_keys=True))

{
  "candidates": [],
  "incumbent_verdict": "FAIL",
  "long_edge_tau": 0.5166663135947416,
  "monotone_decreasing_m": false,
  "rows": [
    {
      "edge_length_max": 1.1132296283665148,
      "edge_length_mean": 0.23865580392143798,
      "edge_length_p99": 0.48702121487636185,
      "fit_key": "9db36086f7472619",
      "geo_ambient_p90": 3.2383827437621684,
      "geo_ambient_ratio": 2.8287271905743148,
      "geo_median": 1.593137593779796,
      "k": 5,
      "lambda_max_pos": 5432.086119993919,
      "lambda_min_neg": -327.62126149676016,
      "long_edge_fraction": 0.00654,
      "m": 0.40643326758373566,
      "n_connected_components": 1,
      "n_edges": 50000,
      "n_negative": 5028,
      "n_positive": 4972,
      "noise_floor": 1.206165416432796e-08,
      "r": 0.06031223626791964,
      "verdict": "FAIL"
    },
    {
      "edge_length_max": 1.160487564708043,
      "edge_length_mean": 0.2480554817989961,
      "edge_length_p99": 0.504291830804723,
      "fit_key": "9fb